# Serving — Export Gold -> CSV para Power BI
**LogiLake | D'LOGIA** — Capa Serving del Medallion Architecture

Este notebook exporta las tablas Gold como archivos CSV limpios,
listos para importar en Power BI Desktop o Power BI Service.

Flujo:
1. Leer tablas Delta Gold desde `./data/gold/`
2. Exportar como CSV single-file con pandas (`UTF-8`, header incluido)
3. Archivos disponibles en `./powerbi/data/` para Power BI

Ver instrucciones de conexion en `powerbi/README.md`.

In [ ]:
# ── SparkSession con Delta Lake 3.1.0 (PySpark 3.5.0) ────────────────────────
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_serving")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

In [ ]:
# ── Rutas locales ─────────────────────────────────────────────────────────────
import os
from pyspark.sql import functions as F

BASE_PATH   = "./data"
GOLD_PATH   = f"{BASE_PATH}/gold"
EXPORT_PATH = "./powerbi/data"

os.makedirs(EXPORT_PATH, exist_ok=True)
print(f"Gold path:   {GOLD_PATH}")
print(f"Export path: {EXPORT_PATH}")

GOLD_TABLES = [
    "kpi_global",
    "kpi_monthly",
    "kpi_category",
    "kpi_nps",
    "kpi_seller_state",
]

In [ ]:
# ── Exportar Gold -> CSV con pandas ──────────────────────────────────────────
# Se usa toPandas() para generar un CSV single-file limpio.
# Ventaja sobre coalesce(1).write.csv(): produce un archivo directamente
# sin subdirectorio, listo para importar en Power BI sin renombrar.

def export_to_csv(table_name: str) -> None:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table_name}")

    # Convertir timestamps a string ISO para compatibilidad con Power BI
    for field in df.schema.fields:
        if str(field.dataType) in ("TimestampType()", "DateType()"):
            df = df.withColumn(field.name, F.col(field.name).cast("string"))

    out_file = f"{EXPORT_PATH}/{table_name}.csv"
    pdf = df.toPandas()
    pdf.to_csv(out_file, index=False, encoding="utf-8")

    size_kb = os.path.getsize(out_file) / 1024
    print(f"OK  {table_name:<25s}  {len(pdf):>6} filas  {size_kb:6.1f} KB  ->  {out_file}")


print("Exportando tablas Gold -> CSV...\n")
for table in GOLD_TABLES:
    try:
        export_to_csv(table)
    except Exception as e:
        print(f"ERR {table}: {e}")

print("\nExportacion completada.")
print(f"Archivos listos en: {os.path.abspath(EXPORT_PATH)}")

In [ ]:
# ── Resumen de archivos exportados ───────────────────────────────────────────
print(f"{'Archivo':<30} {'Tamanio':>10} {'Filas':>8}")
print("-" * 52)
for table in GOLD_TABLES:
    csv_path = f"{EXPORT_PATH}/{table}.csv"
    if os.path.exists(csv_path):
        import pandas as pd
        row_count = len(pd.read_csv(csv_path))
        size_kb   = os.path.getsize(csv_path) / 1024
        print(f"{table + '.csv':<30} {size_kb:>9.1f}KB {row_count:>8}")
    else:
        print(f"{table + '.csv':<30} {'NO ENCONTRADO':>20}")

print("\nSiguiente paso: abrir Power BI Desktop")
print("  Inicio -> Obtener datos -> Carpeta ->" , os.path.abspath(EXPORT_PATH))

In [ ]:
# ── Preview kpi_global.csv ────────────────────────────────────────────────────
import pandas as pd

df_preview = pd.read_csv(f"{EXPORT_PATH}/kpi_global.csv")
print("Schema de kpi_global.csv:")
print(df_preview.dtypes)
print("\nDatos:")
print(df_preview.to_string(index=False))